# Federated Safe Causal World Models (FSCWM) - Main Notebook

This notebook demonstrates the training and evaluation of the Federated Safe Causal World Models (FSCWM) framework. It covers data generation, federated training loops, and visualization of results.


In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import os
from datetime import datetime

from src.config import Config
from src.utils import (
    set_seed,
    Logger,
    Checkpointing,
    FederatedAggregator,
    CausalGraphLearner,
)
from src.data import (
    FederatedReplayBuffer,
    FederatedSafeCausalMountainCar,
    ClientDataset,
)
from src.model import RSSMCore, CausalWorldModel, PolicyNetwork, SafetyCritic
from src.losses import world_model_loss, policy_loss, safety_loss

# --- Configuration and Setup ---
config = Config()
set_seed(config.seed)

print(f"Using device: {config.device}")

# Create directories for saving results and pictures
results_dir = os.path.join(config.save_dir, datetime.now().strftime("%Y%m%d-%H%M%S"))
os.makedirs(results_dir, exist_ok=True)
os.makedirs("../pictures", exist_ok=True)  # For saving plots

logger = Logger(log_dir=results_dir, experiment_name="fscwm_experiment")
checkpointing = Checkpointing(save_dir=results_dir)

In [ ]:
# --- Environment and Data Buffer Initialization ---
client_envs = [
    FederatedSafeCausalMountainCar(config, client_id=i) for i in range(config.n_clients)
]
replay_buffer = FederatedReplayBuffer(
    capacity=config.replay_buffer_size, n_clients=config.n_clients, seed=config.seed
)

print(f"Initialized {config.n_clients} client environments.")
print(f"Replay buffer created with capacity {config.replay_buffer_size}.")

In [ ]:
# --- Model Initialization and Optimizers ---
world_model = CausalWorldModel(config).to(config.device)
policy_network = PolicyNetwork(config).to(config.device)
safety_critic = SafetyCritic(config).to(config.device)

# Optimizers
world_model_optimizer = torch.optim.Adam(
    world_model.parameters(), lr=config.model_learning_rate
)
policy_optimizer = torch.optim.Adam(
    policy_network.parameters(), lr=config.learning_rate_actor
)
safety_critic_optimizer = torch.optim.Adam(
    safety_critic.parameters(), lr=config.learning_rate_critic
)

federated_aggregator = FederatedAggregator(config)

print("Models and optimizers initialized.")

In [ ]:
# --- Federated Training Loop ---

for federated_round in range(config.federated_rounds):
    print(f"\nFederated Round {federated_round + 1}/{config.federated_rounds}")

    # Client Selection
    num_participating_clients = max(
        1, int(config.n_clients * config.client_participation_rate)
    )
    selected_client_ids = np.random.choice(
        range(config.n_clients), num_participating_clients, replace=False
    )

    client_model_updates = []
    client_policy_updates = []
    client_safety_critic_updates = []

    for client_id in selected_client_ids:
        print(f"  Training on Client {client_id}")
        client_env = client_envs[client_id]

        # --- Client-side Data Collection ---
        obs, info = client_env.reset()
        current_h = torch.zeros(1, config.rssm_deterministic_dim, device=config.device)
        current_z = torch.zeros(1, config.rssm_stochastic_dim, device=config.device)

        # Collect experience for this client
        for _ in range(config.n_steps_per_episode):
            action_dist, _ = policy_network(current_h, current_z)
            action = action_dist.sample()

            next_obs, reward, cost, terminated, truncated, info_env = client_env.step(
                action.cpu().numpy()
            )

            # Store experience in client's replay buffer
            replay_buffer.add(
                client_id,
                (
                    obs,
                    action.cpu().numpy(),
                    reward,
                    cost,
                    next_obs,
                    float(terminated),
                    float(truncated),
                ),
            )

            # Update world model states (h, z) for next step (inference)
            obs_tensor = (
                torch.tensor(obs, dtype=torch.float32).unsqueeze(0).to(config.device)
            )
            action_tensor = action.unsqueeze(0).to(config.device)

            with torch.no_grad():
                current_h, current_z, _, _, _, _, _ = world_model.rssm_core(
                    current_h, current_z, action_tensor, obs_tensor
                )

            obs = next_obs
            if terminated or truncated:
                obs, info_env = client_env.reset()
                current_h = torch.zeros(
                    1, config.rssm_deterministic_dim, device=config.device
                )
                current_z = torch.zeros(
                    1, config.rssm_stochastic_dim, device=config.device
                )

        # --- Local World Model and Policy Training ---
        if len(replay_buffer.buffers[client_id]) < config.client_batch_size:
            print(
                f"    Client {client_id}: Not enough data for local training. Skipping."
            )
            continue

        for epoch in range(config.client_epochs):
            client_experiences = replay_buffer.sample(
                client_id, config.client_batch_size
            )
            if not client_experiences:
                break

            (
                obs_batch,
                actions_batch,
                rewards_batch,
                costs_batch,
                next_obs_batch,
                terminateds_batch,
                truncateds_batch,
            ) = ClientDataset.collate_fn(client_experiences)

            # Transfer to device
            (
                obs_batch,
                actions_batch,
                rewards_batch,
                costs_batch,
                next_obs_batch,
                terminateds_batch,
            ) = (
                obs_batch.to(config.device),
                actions_batch.to(config.device),
                rewards_batch.to(config.device),
                costs_batch.to(config.device),
                next_obs_batch.to(config.device),
                terminateds_batch.to(config.device),
            )

            # --- World Model Update (CausalWorldModel) ---
            h_initial = torch.zeros(
                obs_batch.size(0), config.rssm_deterministic_dim, device=config.device
            )
            z_initial = torch.zeros(
                obs_batch.size(0), config.rssm_stochastic_dim, device=config.device
            )

            # Unroll RSSM for a single step for simplicity in this example
            # A full implementation would unroll over a sequence of observations
            (
                h_t,
                z_t_posterior,
                prior_dist,
                posterior_dist,
                recon_obs,
                recon_reward,
                recon_cost,
                causal_reg_loss,
            ) = world_model(
                h_initial,
                z_initial,
                actions_batch,
                obs_batch,
                update_causal_graph=(epoch == 0),
            )

            wm_total_loss, wm_metrics = world_model_loss(
                recon_obs,
                obs_batch,
                recon_reward,
                rewards_batch,
                recon_cost,
                costs_batch,
                prior_dist,
                posterior_dist,
                config,
                causal_regularization_loss=causal_reg_loss,
            )

            world_model_optimizer.zero_grad()
            wm_total_loss.backward()
            torch.nn.utils.clip_grad_norm_(
                world_model.parameters(), config.grad_clip_norm
            )
            world_model_optimizer.step()

            logger.log(
                f"client_{client_id}/wm_loss",
                wm_total_loss.item(),
                federated_round * config.client_epochs + epoch,
            )
            for metric_name, metric_value in wm_metrics.items():
                logger.log(
                    f"client_{client_id}/wm_{metric_name}",
                    metric_value.item(),
                    federated_round * config.client_epochs + epoch,
                )

            # --- Policy and Safety Critic Update (Imagination-based) ---
            # Generate imagined trajectories from the world model
            imagined_h = h_t.detach()
            imagined_z = z_t_posterior.detach()

            imagined_rewards = []
            imagined_costs = []
            imagined_actions_dist = []
            imagined_values = []

            for t_imagine in range(config.horizon):
                action_dist_imagine, value_imagine = policy_network(
                    imagined_h, imagined_z
                )
                action_imagine = action_dist_imagine.sample()

                imagined_h, imagined_z, r_imagine, c_imagine, _ = (
                    world_model.imagine_step(imagined_h, imagined_z, action_imagine)
                )
                imagined_rewards.append(r_imagine)
                imagined_costs.append(c_imagine)
                imagined_actions_dist.append(action_dist_imagine)
                imagined_values.append(value_imagine)

            imagined_rewards = torch.cat(imagined_rewards, dim=-1)
            imagined_costs = torch.cat(imagined_costs, dim=-1)
            imagined_values = torch.cat(imagined_values, dim=-1)

            # Calculate target values for policy and safety critic
            # This is a simplified GAE / Retrace style target for imagined trajectories
            target_values = imagined_rewards.sum(
                dim=-1, keepdim=True
            )  # Simple sum for now
            target_costs = imagined_costs.sum(dim=-1, keepdim=True)

            # Policy and Value Loss
            # Here, we need to choose one action_dist from the imagined_actions_dist list
            # For simplicity, let's take the first one.
            # A full implementation would consider the sequence or sum up log_probs.
            policy_grad_loss, value_pred_loss = policy_loss(
                imagined_actions_dist[0], imagined_values[:, 0], target_values
            )

            # Safety Critic Loss and Constraint Violation
            safety_critic_pred_cost = safety_critic(
                h_t.detach(), z_t_posterior.detach()
            )
            sc_loss, constraint_viol = safety_loss(
                safety_critic_pred_cost, target_costs, config.safety_threshold
            )

            # Combined Policy Optimization with Safety Constraint
            # This is a simplified CPO-like update. A true CPO involves dual variables.
            policy_total_loss = (
                policy_grad_loss
                + config.kl_loss_scale * value_pred_loss
                + constraint_viol
            )  # Add constraint violation directly

            # Update Policy Network
            policy_optimizer.zero_grad()
            policy_total_loss.backward()
            torch.nn.utils.clip_grad_norm_(
                policy_network.parameters(), config.grad_clip_norm
            )
            policy_optimizer.step()

            # Update Safety Critic
            safety_critic_optimizer.zero_grad()
            sc_loss.backward()
            torch.nn.utils.clip_grad_norm_(
                safety_critic.parameters(), config.grad_clip_norm
            )
            safety_critic_optimizer.step()

            logger.log(
                f"client_{client_id}/policy_loss",
                policy_total_loss.item(),
                federated_round * config.client_epochs + epoch,
            )
            logger.log(
                f"client_{client_id}/value_loss",
                value_pred_loss.item(),
                federated_round * config.client_epochs + epoch,
            )
            logger.log(
                f"client_{client_id}/safety_critic_loss",
                sc_loss.item(),
                federated_round * config.client_epochs + epoch,
            )
            logger.log(
                f"client_{client_id}/constraint_violation",
                constraint_viol.item(),
                federated_round * config.client_epochs + epoch,
            )
            logger.log(
                f"client_{client_id}/mean_imagined_reward",
                imagined_rewards.mean().item(),
                federated_round * config.client_epochs + epoch,
            )
            logger.log(
                f"client_{client_id}/mean_imagined_cost",
                imagined_costs.mean().item(),
                federated_round * config.client_epochs + epoch,
            )

        # Collect client updates
        client_model_updates.append(
            {k: v.detach().cpu() for k, v in world_model.state_dict().items()}
        )
        client_policy_updates.append(
            {k: v.detach().cpu() for k, v in policy_network.state_dict().items()}
        )
        client_safety_critic_updates.append(
            {k: v.detach().cpu() for k, v in safety_critic.state_dict().items()}
        )

    # --- Server Aggregation ---
    if client_model_updates:
        global_wm_weights = federated_aggregator.aggregate_model_weights(
            client_model_updates
        )
        global_wm_weights = federated_aggregator.apply_differential_privacy(
            global_wm_weights
        )
        world_model.load_state_dict(global_wm_weights)

        global_policy_weights = federated_aggregator.aggregate_model_weights(
            client_policy_updates
        )
        global_policy_weights = federated_aggregator.apply_differential_privacy(
            global_policy_weights
        )
        policy_network.load_state_dict(global_policy_weights)

        global_safety_critic_weights = federated_aggregator.aggregate_model_weights(
            client_safety_critic_updates
        )
        global_safety_critic_weights = federated_aggregator.apply_differential_privacy(
            global_safety_critic_weights
        )
        safety_critic.load_state_dict(global_safety_critic_weights)

        print(f"Federated Round {federated_round + 1}: Global models aggregated.")
    else:
        print(f"Federated Round {federated_round + 1}: No client updates to aggregate.")

    # Checkpointing
    if (federated_round + 1) % config.checkpoint_interval == 0:
        checkpointing.save_checkpoint(
            world_model, world_model_optimizer, federated_round, prefix="wm"
        )
        checkpointing.save_checkpoint(
            policy_network, policy_optimizer, federated_round, prefix="policy"
        )
        checkpointing.save_checkpoint(
            safety_critic, safety_critic_optimizer, federated_round, prefix="safety"
        )

print("\nFederated training completed!")
logger.close()

In [ ]:
# --- Visualization of Results ---

print("\nGenerating plots...")

# Example: Plotting World Model Loss
plt.figure(figsize=(10, 6))
wm_losses = logger.get_metrics().get(
    "client_0/wm_loss", []
)  # Assuming we plot client 0's loss for now
steps = [s for s, _ in wm_losses]
values = [v for _, v in wm_losses]
plt.plot(steps, values)
plt.xlabel("Training Step")
plt.ylabel("World Model Loss")
plt.title("World Model Loss over Training Steps (Client 0)")
plt.grid(True)
plt.savefig("../pictures/fig_01_wm_loss.png", dpi=300)
plt.show()

# Example: Plotting Policy Loss
plt.figure(figsize=(10, 6))
policy_losses = logger.get_metrics().get("client_0/policy_loss", [])
steps = [s for s, _ in policy_losses]
values = [v for _, v in policy_losses]
plt.plot(steps, values)
plt.xlabel("Training Step")
plt.ylabel("Policy Loss")
plt.title("Policy Loss over Training Steps (Client 0)")
plt.grid(True)
plt.savefig("../pictures/fig_02_policy_loss.png", dpi=300)
plt.show()

# Example: Plotting Mean Imagined Reward
plt.figure(figsize=(10, 6))
imagined_rewards = logger.get_metrics().get("client_0/mean_imagined_reward", [])
steps = [s for s, _ in imagined_rewards]
values = [v for _, v in imagined_rewards]
plt.plot(steps, values)
plt.xlabel("Training Step")
plt.ylabel("Mean Imagined Reward")
plt.title("Mean Imagined Reward over Training Steps (Client 0)")
plt.grid(True)
plt.savefig("../pictures/fig_03_mean_reward.png", dpi=300)
plt.show()

# Example: Plotting Mean Imagined Cost
plt.figure(figsize=(10, 6))
imagined_costs = logger.get_metrics().get("client_0/mean_imagined_cost", [])
steps = [s for s, _ in imagined_costs]
values = [v for _, v in imagined_costs]
plt.plot(steps, values)
plt.xlabel("Training Step")
plt.ylabel("Mean Imagined Cost")
plt.title("Mean Imagined Cost over Training Steps (Client 0)")
plt.grid(True)
plt.savefig("../pictures/fig_04_mean_cost.png", dpi=300)
plt.show()

print("Plots generated and saved to ../pictures/ directory.")

In [ ]:
# --- Model Initialization and Optimizers ---
world_model = CausalWorldModel(config).to(config.device)
policy_network = PolicyNetwork(config).to(config.device)
safety_critic = SafetyCritic(config).to(config.device)

# Optimizers
world_model_optimizer = torch.optim.Adam(
    world_model.parameters(), lr=config.model_learning_rate
)
policy_optimizer = torch.optim.Adam(
    policy_network.parameters(), lr=config.learning_rate_actor
)
safety_critic_optimizer = torch.optim.Adam(
    safety_critic.parameters(), lr=config.learning_rate_critic
)

federated_aggregator = FederatedAggregator(config)

print("Models and optimizers initialized.")